# Project: Analyzing Car Reviews with LLMs
**Module 05 – Developing Large Language Models**

> Apply LLM techniques to a real-world car review sentiment and entity analysis task.

## Project Overview

In this project, you will:
1. **Load and preprocess** a car reviews dataset
2. **Perform sentiment analysis** using a pre-trained LLM
3. **Extract key entities** (car brands, features)
4. **Summarize** long reviews
5. **Evaluate** model performance with standard metrics

### Skills Practiced
- HuggingFace `pipeline` for NLP tasks
- Batch inference on real text data
- Evaluation with `sklearn` and `evaluate`

## Step 1: Load and Explore the Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Sample car review data (replace with real CSV)
reviews_data = {
    'review': [
        "The Toyota Camry is an excellent family sedan with great fuel economy.",
        "I'm very disappointed with my Honda Civic. Constant issues from day one.",
        "The Tesla Model 3 is absolutely revolutionary. Best car I've owned!",
        "Ford F-150 is reliable but nothing special. Average fuel efficiency.",
        "BMW 3 Series delivers incredible driving dynamics and a premium feel.",
        "The Hyundai Elantra is surprisingly good for the price. Recommended!",
    ],
    'true_sentiment': [1, 0, 1, 0, 1, 1],
    'brand': ['Toyota', 'Honda', 'Tesla', 'Ford', 'BMW', 'Hyundai']
}

df = pd.DataFrame(reviews_data)
print(df[['brand', 'review', 'true_sentiment']].to_string())
print(f"\nTotal reviews: {len(df)}")
print(f"Positive: {df.true_sentiment.sum()}, Negative: {(~df.true_sentiment.astype(bool)).sum()}")

## Step 2: Sentiment Analysis with LLM

In [ ]:
from transformers import pipeline

# Load sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Run inference on all reviews
sentiment_results = sentiment_pipeline(df['review'].tolist())

# Map results to binary labels
df['predicted_sentiment'] = [
    1 if r['label'] == 'POSITIVE' else 0 for r in sentiment_results
]
df['confidence'] = [r['score'] for r in sentiment_results]

print(df[['brand', 'true_sentiment', 'predicted_sentiment', 'confidence']].to_string())

from sklearn.metrics import accuracy_score, classification_report
acc = accuracy_score(df['true_sentiment'], df['predicted_sentiment'])
print(f"\nAccuracy: {acc:.2%}")
print("\nDetailed Report:")
print(classification_report(df['true_sentiment'], df['predicted_sentiment'],
                            target_names=['Negative', 'Positive']))

## Step 3: Summarize Long Reviews

In [ ]:
# Summarization for longer reviews
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

long_review = """
I have owned my Toyota Camry for three years now and I have to say it has been one
of the best decisions I have ever made. The fuel economy is outstanding, averaging
around 35 miles per gallon on the highway. The interior quality has held up well,
and I have had zero mechanical issues. The infotainment system was a bit dated at
first but a software update resolved most of the concerns. Overall comfort is
excellent for long road trips. I would highly recommend this car to any family
looking for a reliable, economical sedan.
"""

summary = summarizer(long_review, max_length=60, min_length=20, do_sample=False)
print("Original review length:", len(long_review.split()), "words")
print("\nSummary:", summary[0]['summary_text'])

## Step 4: Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sentiment distribution
axes[0].bar(['Negative', 'Positive'],
            [df['predicted_sentiment'].eq(0).sum(),
             df['predicted_sentiment'].eq(1).sum()],
            color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Predicted Sentiment Distribution')
axes[0].set_ylabel('Count')

# Confidence scores
colors = ['#2ecc71' if s == 1 else '#e74c3c' for s in df['predicted_sentiment']]
axes[1].barh(df['brand'], df['confidence'], color=colors)
axes[1].set_title('Confidence Score by Brand')
axes[1].set_xlabel('Confidence')
axes[1].set_xlim(0, 1)
axes[1].axvline(0.5, linestyle='--', color='gray', alpha=0.5)

plt.tight_layout()
plt.show()

## Project Conclusions

- Pre-trained LLMs can achieve **high accuracy** on sentiment analysis with **zero training**
- `pipeline()` makes it easy to apply LLMs to real datasets
- Confidence scores help identify borderline predictions
- Summarization compresses lengthy reviews into actionable insights

### Next Steps
- Fine-tune BERT on a larger car reviews dataset
- Add **Named Entity Recognition (NER)** to extract car features
- Build an end-to-end review analysis dashboard